# APIM ❤️ AI Foundry

## Access Controlling lab
![flow](../../images/access-controlling.gif)

Playground to try the [OAuth 2.0 authorization feature](https://learn.microsoft.com/azure/api-management/api-management-authenticate-authorize-azure-openai#oauth-20-authorization-using-identity-provider) using identity provider to enable more fine-grained access to AI Foundry models by particular users or client.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`... 


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management) 
- Adjust the models and versions according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models) 

In [1]:
import os, sys, json, random
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}-7152" # change the name to match your naming style
resource_group_location = "eastus2"

aiservices_config = [{"name": "foundry1", "location": "eastus2"}]

models_config = [{"name": "gpt-4o-mini", "publisher": "OpenAI", "version": "2024-07-18", "sku": "GlobalStandard", "capacity": 100}]

apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

inference_api_path = "inference"  # path to the inference API in the APIM service
inference_api_type = "AzureOpenAI"  # options: AzureOpenAI, AzureAI, OpenAI, PassThrough
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

app_registration_name = f"{deployment_name}-app" # name of the app that will be registered in Microsoft Entra ID

utils.print_ok('Notebook initialized')


✅ Notebook initialized ⌚ 14:15:11.811967 


<a id='1'></a>
### 1️⃣ Create the App Registration in Microsoft Entra ID
The following command creates a client application registration

In [2]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")


if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 14:15:52.624931 :0s]
👉🏽 Current user: baoqger@gmail.com
👉🏽 Tenant ID: 293882e2-ff46-4747-b46a-a84ca8e98fa0
👉🏽 Subscription ID: 784e6c68-d702-4a8b-a678-2e0f2f36deab


In [3]:

client_id = None
output = utils.run(f"az ad app list --filter \"displayName eq '{app_registration_name}'\"", f"Retrieved app registration with name {app_registration_name}", "Failed to get the app registration")
if output.success and output.json_data:
    client_id = output.json_data[0]['appId']
else:
    output = utils.run(f"az ad app create --display-name {app_registration_name} --is-fallback-public-client true", f"Created app registration with name {app_registration_name}", "Failed to create the app registration")
    if output.success and output.json_data:
        client_id = output.json_data['appId']

print(f"👉🏻 Client Id: {client_id}")


⚙️ Running: az ad app list --filter "displayName eq 'access-controlling-app'" 
✅ Retrieved app registration with name access-controlling-app ⌚ 14:16:10.631507 :3s]
👉🏻 Client Id: 34e82664-380a-4455-aff6-16de6096e022


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.


In [4]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

⚙️ Running: az group show --name lab-access-controlling-7152 
👉🏽 Resource group lab-access-controlling-7152 does not yet exist. Creating the resource group now...
⚙️ Running: az group create --name lab-access-controlling-7152 --location eastus2 --tags source=ai-gateway 
✅ Resource group 'lab-access-controlling-7152' created ⌚ 14:20:02.427055 :4s]


In [5]:


# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name },
        "tenantId": { "value": tenant_id },
        "clientId": { "value": client_id }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))



In [6]:
# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

⚙️ Running: az deployment group create --name access-controlling --resource-group lab-access-controlling-7152 --template-file main.bicep --parameters params.json 
✅ Deployment 'access-controlling' succeeded ⌚ 14:24:33.297270 :7s]


<a id='4'></a>
### 4️⃣ Get the deployment outputs

We are now at the stage where we only need to retrieve the gateway URL and the subscription before we are ready for testing.

In [7]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get("key") # default api key to the first subscription key

⚙️ Running: az deployment group show --name access-controlling -g lab-access-controlling-7152 
✅ Retrieved deployment: access-controlling ⌚ 14:27:22.347015 :2s]
👉🏽 Log Analytics Id: d65e69fb-d2b9-4181-82c3-1c91116bd9c2
👉🏽 APIM Service Id: /subscriptions/784e6c68-d702-4a8b-a678-2e0f2f36deab/resourceGroups/lab-access-controlling-7152/providers/Microsoft.ApiManagement/service/apim-phjdx65rodb4g
👉🏽 APIM API Gateway URL: https://apim-phjdx65rodb4g.azure-api.net
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****2b8f


### Get a general User Token

In [8]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
credential = DefaultAzureCredential()
token = credential.get_token("https://management.azure.com/.default").token
print(token)


eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIng1dCI6InJ5QngtNjVqTm94SDRGUUhfNnphck5qYU5mNCIsImtpZCI6InJ5QngtNjVqTm94SDRGUUhfNnphck5qYU5mNCJ9.eyJhdWQiOiJodHRwczovL21hbmFnZW1lbnQuYXp1cmUuY29tIiwiaXNzIjoiaHR0cHM6Ly9zdHMud2luZG93cy5uZXQvMjkzODgyZTItZmY0Ni00NzQ3LWI0NmEtYTg0Y2E4ZTk4ZmEwLyIsImlhdCI6MTc5MzQyNzgwMiwibmJmIjoxNzkzNDI3ODAyLCJleHAiOjE3OTM0MzIyNzgsImFjciI6IjEiLCJhY3JzIjpbInAxIl0sImFpbyI6IkFhUUFXLzhlQUFBQU10K2pIb0d2VUNhRm53bXBtdWNQYW5GSHJISGFNYzEvWTk0TUx3dTFOaVYyeEg5U2RuaWsxSWhtU2hPalMwQkZLa0JlY092dFgzbU4yT2ExcDIzRC9NUHhLYjl3dWh5bjJ0TW9YdHdSd0xmbjR2eVdiakZXdytLdDBveEFMWTduMFdIcGo0Tmc4Zmk4RmxWM0FoSGc5UnJTanRjWHkwLzJNamx5dUhIV0thTDM2bmE5dGlSdmRSMTQyTjlOSDM3Mm9VZ1RYZHJSVlhzZXQ5aEo1TEg5U1E9PSIsImFsdHNlY2lkIjoiMTpsaXZlLmNvbTowMDAzQkZGRTk4RjMyNjk3IiwiYW1yIjpbInB3ZCIsIm1mYSJdLCJhcHBpZCI6IjA0YjA3Nzk1LThkZGItNDYxYS1iYmVlLTAyZjllMWJmN2I0NiIsImFwcGlkYWNyIjoiMCIsImVtYWlsIjoiYmFvcWdlckBnbWFpbC5jb20iLCJmYW1pbHlfbmFtZSI6IkJhbyIsImdpdmVuX25hbWUiOiJDaHJpcyIsImdyb3VwcyI6WyI0ODA2MjkyMi02YzhlLTQ5ZmItYjVhZC1iZTQ

<a id='5'></a>
### 5️⃣ Create a device flow to get the access token

Notes for fine grained authorization:
- The APIM [JWT validation policy](https://learn.microsoft.com/azure/api-management/validate-azure-ad-token-policy) can check for specific claims (that needs to exist in the token) and apply fine-grained authorization.
- Group claims is a typical method. You can use this approach to drive authorization. However, when the user is a member of too many groups, the `groups` will be excluded from the token due to limitations in token size.
- An alternative is to configure app role definitions and assign users/groups to app roles. This Zero Trust developer best practice improves flexibility and control while increasing application security with least privilege. [Learn more](https://learn.microsoft.com/security/zero-trust/develop/configure-tokens-group-claims-app-roles).
- To obtain the `roles` claim, navigate to the "Expose an API" section of the App Registration. Add the Application ID URI and a scope. Then, copy the full scope `(app://<id>/scope)` and replace in the scope array below.
- Navigate to the "App Roles" blade and create an App Role (ex: OpenAI.ChatCompletion) for Users/Groups members. Then assign the testing user or group to the App Role.   
- After logging in, use https://jwt.io/ to decode the `access_token` variable and verify that the `roles` are being sent.
- With the above configuration, you can add the following fragment to the APIM policy to verify that the user belongs to a specific App Role:
```
            <required-claims>
                <claim name="roles" match="any">
                    <value>OpenAI.ChatCompletion</value>
                </claim>
            </required-claims>
```



### Start the Devide Code Flow 

In [ ]:
import json
import msal

app = msal.PublicClientApplication(client_id, authority = "https://login.microsoftonline.com/" + tenant_id)

flow = app.initiate_device_flow(scopes = [
    "api://34e82664-380a-4455-aff6-16de6096e022/chatcomplete"
    ])

if "user_code" not in flow:
    raise ValueError(
        "Fail to create device flow. Err: %s" % json.dumps(flow, indent = 4))

print(flow["message"])

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code SVSBJ8J82 to authenticate.


<a id='6'></a>
### 6️⃣ Acquire the access token

In [53]:
import requests, base64, json

result = app.acquire_token_by_device_flow(flow)

if "access_token" in result:
    access_token = result['access_token']
    graph_data = requests.get(  # Use token to call downstream service
        "https://graph.microsoft.com/v1.0/me",
        headers={'Authorization': 'Bearer ' + access_token},).json()
    print("Graph API call result: %s" % json.dumps(graph_data, indent = 2))
    # print(access_token) # Use a tool like https://jwt.io/ to decode the access token and see its contents
else:
    print(result.get("error"))
    print(result.get("error_description"))
    print(result.get("correlation_id"))  # You may need this when reporting a bug



Graph API call result: {
  "error": {
    "code": "InvalidAuthenticationToken",
    "message": "Access token validation failure. Invalid audience.",
    "innerError": {
      "date": "2026-10-31T14:02:38",
      "request-id": "a72f3da8-7e5a-47e4-b140-be81cff8b708",
      "client-request-id": "a72f3da8-7e5a-47e4-b140-be81cff8b708"
    }
  }
}


<a id='requests'></a>
### 🧪 Test Access Control

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to debug the policy.

In [36]:
import requests, base64, json

url = f"{apim_resource_gateway_url}/{inference_api_path}/openai/deployments/{models_config[0]['name']}/chat/completions?api-version={inference_api_version}"

print(url)
def access_control_test(token, prompt):
    messages = {"messages": [
        {"role": "system", "content": "You are a sarcastic unhelpful assistant."},
        {"role": "user", "content": prompt}
    ]}

    response = requests.post(url, headers={'api-key': api_key, 'Authorization': 'Bearer ' + token}, json=messages)
    utils.print_response_code(response)

    if response.status_code == 200:
        data = json.loads(response.text)
        print(f"💬 {data.get('choices')[0].get('message').get('content')}")
    else:
        print(response.text)





https://apim-phjdx65rodb4g.azure-api.net/inference/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-03-01-preview


In [20]:
import requests, base64, json


def decode_access_token(token):
    header, payload, signature = token.split('.')

    def pad(b):
        return b + '=' * (-len(b) % 4)

    print("Decoded JWT Header and Payload:")
    print(json.dumps(json.loads(base64.urlsafe_b64decode(pad(header)).decode('utf-8')), indent=4))
    print(json.dumps(json.loads(base64.urlsafe_b64decode(pad(payload)).decode('utf-8')), indent=4))

In [21]:
# Test with a general access token issued for the user
access_control_test(token, "Can you tell me the time, please?")

Response status: 401 - Unauthorized
{ "statusCode": 401, "message": "Invalid Azure AD JWT" }


In [22]:
# Decode the access token to see its contents
decode_access_token(token)

Decoded JWT Header and Payload:
{
    "typ": "JWT",
    "alg": "RS256",
    "x5t": "ryBx-65jNoxH4FQH_6zarNjaNf4",
    "kid": "ryBx-65jNoxH4FQH_6zarNjaNf4"
}
{
    "aud": "https://management.azure.com",
    "iss": "https://sts.windows.net/293882e2-ff46-4747-b46a-a84ca8e98fa0/",
    "iat": 1793427802,
    "nbf": 1793427802,
    "exp": 1793432278,
    "acr": "1",
    "acrs": [
        "p1"
    ],
    "aio": "AaQAW/8eAAAAMt+jHoGvUCaFnwmpmucPanFHrHHaMc1/Y94MLwu1NiV2xH9Sdnik1IhmShOjS0BFKkBecOvtX3mN2Oa1p23D/MPxKb9wuhyn2tMoXtwRwLfn4vyWbjFWw+Kt0oxALY7n0WHpj4Ng8fi8FlV3AhHg9RrSjtcXy0/2MjlyuHHWKaL36na9tiRvdR142N9NH372oUgTXdrRVXset9hJ5LH9SQ==",
    "altsecid": "1:live.com:0003BFFE98F32697",
    "amr": [
        "pwd",
        "mfa"
    ],
    "appid": "04b07795-8ddb-461a-bbee-02f9e1bf7b46",
    "appidacr": "0",
    "email": "baoqger@gmail.com",
    "family_name": "Bao",
    "given_name": "Chris",
    "groups": [
        "48062922-6c8e-49fb-b5ad-be402be21486"
    ],
    "idp": "live.com",
    "idtyp

In [58]:
# Test with the access token issued for the agent client
access_control_test(access_token, "Can you tell me the time, please?")

Response status: 401 - Unauthorized
{ "statusCode": 401, "message": "Invalid Azure AD JWT" }


In [54]:
# Decode the access token to see its contents
decode_access_token(access_token)

Decoded JWT Header and Payload:
{
    "typ": "JWT",
    "alg": "RS256",
    "x5t": "ryBx-65jNoxH4FQH_6zarNjaNf4",
    "kid": "ryBx-65jNoxH4FQH_6zarNjaNf4"
}
{
    "aud": "api://34e82664-380a-4455-aff6-16de6096e022",
    "iss": "https://sts.windows.net/293882e2-ff46-4747-b46a-a84ca8e98fa0/",
    "iat": 1793455057,
    "nbf": 1793455057,
    "exp": 1793459014,
    "acr": "1",
    "aio": "AcQAO/8eAAAAcPX6OkwetV5hiq5+2Xhaa3fJZIS6i6N8XTZFzLE0fxTQSynAhBdTlqaF5EHn0CUroG7NaGu+qhyriZj2CooNHjAw/xQIDrTPDqjL+4jr/wyqGqz8omtPM4ZTrQcv7ARMgl8AbZ34aQviH4GLDXEfJn0O8irb9LuTTgDtyRg2Qy9HJ3iqHJqqdRCYJH+mUeASY33/W4p90wnmixXiSwxNBxqLXKIXD0P2m7OYbmWiu8TuH4bIlnImz8FXMHEkY7Dz",
    "amr": [
        "pwd",
        "mfa"
    ],
    "appid": "34e82664-380a-4455-aff6-16de6096e022",
    "appidacr": "0",
    "email": "baoqger@gmail.com",
    "family_name": "Bao",
    "given_name": "Chris",
    "idp": "live.com",
    "ipaddr": "2408:8207:78b2:9f31:1c6f:eba4:79c4:926e",
    "name": "Chris Bao",
    "oid": "d672a57e-86cc